# 01 · Sampling, blur & resolution
**Biomedical Imaging Interactive Labs · 30–45 minutes · No coding required**

How can a detector show clear stripes that were never in the object? Explore the image-formation chain:

**Intensity target → optical blur → pixel-area averaging → sampling**

By the end, you should be able to distinguish blur from coarse sampling, predict an alias frequency, and use an MTF curve to explain loss of contrast.

### Start here
1. Choose **Runtime → Run all**. A CPU runtime is sufficient; wait until the explorer appears.
2. Scroll to **Resolution explorer**. Release a slider to update. Try the four preset buttons.
3. Answer the five questions below using the controls and measurements. No Python edits are needed.

Code is collapsed in Colab; open a code cell's form menu to view it if desired. All lengths are in **mm in the image plane**, with unit magnification. Intensities are normalized from 0 to 1; every image uses the same grayscale limits.


In [ ]:
#@title Setup · run automatically with Run all
import importlib.util
import subprocess
import sys

# Install only missing packages; avoid replacing Colab's working scientific stack.
required = {"numpy": "numpy>=1.26,<3", "scipy": "scipy>=1.11,<2",
            "matplotlib": "matplotlib>=3.8,<4", "ipywidgets": "ipywidgets>=8.1,<9"}
missing = [spec for name, spec in required.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
try:
    from google.colab import output as colab_output
except ImportError:
    pass  # Standard Jupyter uses its built-in widget manager.
else:
    colab_output.enable_custom_widget_manager()

import numpy as np
import scipy
from scipy.optimize import brentq
import matplotlib
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

print("Ready · Python", sys.version.split()[0], "| NumPy", np.__version__,
      "| SciPy", scipy.__version__, "| Matplotlib", matplotlib.__version__,
      "| ipywidgets", widgets.__version__)


## 1 · The physical model
The resolution target is a sinusoidal intensity grating, constant in $y$:
\[
I_0(x)=\tfrac12[1+C_0\cos(2\pi f x+\phi)],\qquad C_0=0.9.
\]
Here $f$ is spatial frequency in **cycles/mm** (one bright–dark pair per cycle), $\phi$ is target phase, and the stripe period is $1/f$ mm. Sinusoids contain one spatial frequency, making contrast transfer especially clear.

**Optical blur.** We approximate an incoherent, linear, shift-invariant imaging system with a normalized Gaussian intensity PSF:
\[
h(x,y)=\frac{1}{2\pi\sigma^2}\exp\!\left[-\frac{x^2+y^2}{2\sigma^2}\right],
\qquad \mathrm{FWHM}=2\sqrt{2\ln 2}\,\sigma\approx2.355\sigma.
\]
Convolution attenuates the sinusoid by $H_{\rm opt}(f)=\exp(-2\pi^2\sigma^2f^2)$. The setting $\sigma=0$ means an ideal delta PSF. This Gaussian is an educational approximation, not an exact diffraction PSF.

**Detector aperture.** Square pixels of pitch and active width $p$ mm average the blurred intensity over their full area (100% fill factor), then report one value at each center $x_n=(n+\tfrac12)p$:
\[
d_n=\frac{1}{p}\int_{x_n-p/2}^{x_n+p/2} I_{\rm blur}(x)\,dx
=\tfrac12[1+C_0H_{\rm opt}(f)\,\mathrm{sinc}(pf)\cos(2\pi f x_n+\phi)].
\]
The $y$ average changes nothing because the target is constant in $y$; $\mathrm{sinc}(u)=\sin(\pi u)/(\pi u)$, with $\mathrm{sinc}(0)=1$. Values are **intensity averages**, not photon counts. Photon noise will be introduced in a later lab.

**Presampling MTF and sampling are different.** The contrast attenuation before sampling is
\[
\mathrm{MTF}_{\rm pre}(f)=|H_{\rm opt}(f)\,\mathrm{sinc}(pf)|,
\qquad f_s=1/p,\qquad f_N=1/(2p).
\]
A negative sinc reverses the sinusoid's phase; MTF retains only its magnitude. Sampling can fold frequencies above $f_N$ into $[0,f_N]$:
\[
f_{\rm alias}=\left|\left((f+f_s/2)\bmod f_s\right)-f_s/2\right|.
\]
For a single sinusoid, **strictly below** $f_N$ permits phase-independent recovery in this ideal model. Exactly at Nyquist, the sampled contrast depends on phase; at integer multiples of $f_s$ the pattern can look constant. The Gaussian has no hard cutoff, so it reduces rather than eliminates aliasing.

We calculate convolution and pixel integration **analytically** for an infinite grating, then crop an 8 × 8 mm field. There is no numerical blur grid or padded image boundary. Nearest-neighbor display shows the recorded pixels without smoothing. The profile's dashed curve is a sinusoid through the samples at the folded frequency; it is a visual aid, not a uniquely recovered object.


In [ ]:
#@title Image-formation model · exact sinusoidal solution
FOV_MM = 8.0
INPUT_CONTRAST = 0.9
PITCHES_MM = (0.05, 0.10, 0.20, 0.25, 0.40, 0.50)

def optical_otf(f, sigma):
    return np.exp(-2 * np.pi**2 * sigma**2 * np.asarray(f)**2)

def presampling_otf(f, sigma, pitch):
    return optical_otf(f, sigma) * np.sinc(pitch * np.asarray(f))

def alias_frequency(f, pitch):
    fs = 1 / pitch
    return abs((f + fs / 2) % fs - fs / 2)

def sample_detector(f, sigma, pitch, phase_deg):
    n = int(round(FOV_MM / pitch))
    centers = (np.arange(n) + 0.5) * pitch
    response = presampling_otf(f, sigma, pitch)
    samples = 0.5 * (1 + INPUT_CONTRAST * response *
                     np.cos(2 * np.pi * f * centers + np.deg2rad(phase_deg)))
    return centers, samples

def fit_sampled_sinusoid(centers, samples, frequency):
    # At DC/Nyquist the sin/cos basis cannot identify both amplitude and phase.
    design = np.column_stack([np.ones_like(centers),
                              np.cos(2 * np.pi * frequency * centers),
                              np.sin(2 * np.pi * frequency * centers)])
    if np.linalg.cond(design) > 1e10:
        return None
    coefficients = np.linalg.lstsq(design, samples, rcond=None)[0]
    contrast = np.hypot(*coefficients[1:]) / coefficients[0]
    return contrast, coefficients

def mtf50(sigma, pitch):
    # Unique crossing on the first lobe (before the aperture's first zero).
    return brentq(lambda f: presampling_otf(f, sigma, pitch) - 0.5, 0, 1 / pitch)

def measurements(f, sigma, pitch, phase_deg):
    x, values = sample_detector(f, sigma, pitch, phase_deg)
    fa = alias_frequency(f, pitch)
    fit = fit_sampled_sinusoid(x, values, fa)
    return dict(x=x, values=values, alias=fa, nyquist=1/(2*pitch),
                pre_mtf=abs(presampling_otf(f, sigma, pitch)),
                mtf50=mtf50(sigma, pitch), fit=fit,
                sample_range_contrast=np.ptp(values)/(values.max()+values.min()))


## 2 · Resolution explorer
**Read left to right:** object intensity, optical image, recorded detector image. The lower panels explain what changed. Images show an 8 mm square; the profile zooms to its first 4 mm.

- **Fine sampling:** baseline with little blur and many pixels per period.
- **Blur limited:** same sampling, but a broader PSF.
- **Aliasing:** a 3 cycles/mm object sampled at 4 samples/mm appears at 1 cycle/mm.
- **Nyquist boundary:** change phase from 0° to 90° and watch a pattern disappear or return.

**Metrics:** MTF50 is the first frequency where presampling contrast transfer falls to 50%. It is a bandwidth indicator, not a hard resolution limit. The sinusoidal fit measures contrast at the folded frequency; it cannot establish whether that frequency belongs to the original object. “Sample range contrast” uses the sampled maximum and minimum and may miss the true sinusoidal extrema.


In [ ]:
#@title Launch resolution explorer · use the controls below
COLORS = dict(optical="#197c81", aperture="#b57921", total="#35356e", alias="#bd3c53")

def make_dashboard(f=1.0, sigma=0.04, pitch=0.10, phase_deg=0.0):
    m = measurements(f, sigma, pitch, phase_deg)
    phi = np.deg2rad(phase_deg)
    fine_x = (np.arange(800) + 0.5) * FOV_MM / 800
    target = 0.5 * (1 + INPUT_CONTRAST * np.cos(2*np.pi*f*fine_x + phi))
    blurred = 0.5 * (1 + INPUT_CONTRAST * optical_otf(f, sigma) *
                     np.cos(2*np.pi*f*fine_x + phi))
    with plt.rc_context({"font.size": 10, "axes.spines.top": False,
                         "axes.spines.right": False, "figure.facecolor": "white"}):
        fig, axes = plt.subplots(2, 3, figsize=(14, 8.4), constrained_layout=True)
        fig.suptitle(f"Resolution explorer  |  f = {f:g} cycles/mm  ·  σ = {sigma:g} mm"
                     f"  ·  p = {pitch:g} mm  ·  phase = {phase_deg:g}°", fontsize=14)
        images = [np.broadcast_to(target, (400, len(target))),
                  np.broadcast_to(blurred, (400, len(blurred))),
                  np.broadcast_to(m["values"], (len(m["values"]), len(m["values"])))]
        titles = ["1 · Object: sinusoidal target", "2 · Optical image: Gaussian blur",
                  f"3 · Detector: {len(m['x'])} × {len(m['x'])} pixels"]
        for ax, image, title in zip(axes[0], images, titles):
            im = ax.imshow(image, cmap="gray", vmin=0, vmax=1, origin="lower",
                           extent=[0, FOV_MM, 0, FOV_MM], interpolation="nearest")
            ax.set(title=title, xlabel="x (mm)", ylabel="y (mm)", xticks=[0, 2, 4, 6, 8])
        fig.colorbar(im, ax=list(axes[0]), shrink=0.75, label="Normalized intensity")

        ax = axes[1, 0]
        ax.plot(fine_x, blurred, color=COLORS["optical"], label="Optical intensity", lw=1.7)
        ax.plot(m["x"], m["values"], "o", ms=3.5, color=COLORS["alias"], label="Pixel averages")
        if m["fit"] is not None:
            _, c = m["fit"]
            folded = c[0] + c[1]*np.cos(2*np.pi*m["alias"]*fine_x) + c[2]*np.sin(2*np.pi*m["alias"]*fine_x)
            ax.plot(fine_x, folded, "--", lw=1.4, color=COLORS["total"], label="Fit at folded frequency")
        ax.set(title="4 · Profile: first 4 mm", xlabel="x (mm)", ylabel="Normalized intensity",
               xlim=(0, 4), ylim=(-0.03, 1.03))
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(alpha=0.2)

        ax = axes[1, 1]
        # Fixed frequency axis: moving pitch does not disguise changes of bandwidth.
        freq = np.linspace(0, 12, 1601)
        ax.plot(freq, optical_otf(freq, sigma), color=COLORS["optical"], label="Optical MTF")
        ax.plot(freq, abs(np.sinc(pitch*freq)), color=COLORS["aperture"], label="Pixel-aperture MTF")
        ax.plot(freq, abs(presampling_otf(freq, sigma, pitch)), color=COLORS["total"], lw=2, label="Combined presampling MTF")
        ax.axvspan(m["nyquist"], 12, color=COLORS["alias"], alpha=0.07)
        ax.axvline(m["nyquist"], color=COLORS["alias"], ls=":", label="Nyquist")
        ax.axvline(f, color="0.4", lw=1, ls="--", label="Target frequency")
        ax.scatter([f], [m["pre_mtf"]], color=COLORS["total"], s=35, zorder=5)
        ax.set(title="5 · Contrast transfer before sampling", xlabel="Spatial frequency (cycles/mm)",
               ylabel="MTF (unitless)", xlim=(0, 12), ylim=(-0.02, 1.04))
        ax.legend(fontsize=7.5, loc="upper right")
        ax.grid(alpha=0.2)

        ax = axes[1, 2]
        psf_x = np.linspace(-1.5, 1.5, 1601)
        if sigma > 0:
            # Normalized central cross-section, not the 2-D PSF's absolute units.
            ax.plot(psf_x, np.exp(-0.5*(psf_x/sigma)**2), color=COLORS["optical"], lw=2)
            half_width = np.sqrt(2*np.log(2))*sigma
            ax.hlines(0.5, -half_width, half_width, color=COLORS["alias"], lw=2, label="Optical FWHM")
        else:
            ax.annotate("Ideal delta PSF", xy=(0, 0.98), xytext=(0.25, 0.7),
                        arrowprops={"arrowstyle":"->"})
            ax.vlines(0, 0, 1, color=COLORS["optical"], lw=2)
        ax.plot([-pitch/2, -pitch/2, pitch/2, pitch/2], [0, 1, 1, 0],
                color=COLORS["aperture"], ls="--", label="Pixel aperture (width p)")
        ax.set(title="6 · PSF and detector aperture", xlabel="Offset x (mm)",
               ylabel="Peak-normalized cross-section", xlim=(-1.5, 1.5), ylim=(0, 1.1))
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(alpha=0.2)
    return fig

def metric_html(f, sigma, pitch, phase_deg):
    m = measurements(f, sigma, pitch, phase_deg)
    if np.isclose(f, m["nyquist"], atol=1e-10, rtol=0):
        status = "NYQUIST BOUNDARY · recorded contrast depends on phase"
    elif f > m["nyquist"]:
        status = "ABOVE NYQUIST · a surviving signal folds to a lower frequency"
    else:
        status = "BELOW NYQUIST · target frequency is within the sampling band"
    fit_text = (f"{m['fit'][0]:.4f}" if m["fit"] is not None else
                "indeterminate (DC or Nyquist; amplitude/phase cannot be separated)")
    sign_text = "yes (negative aperture response)" if presampling_otf(f, sigma, pitch) < -1e-12 else "no"
    rows = [
        ("Target period", f"{1/f:.3f} mm"),
        ("Pixels per target period", f"{1/(pitch*f):.3f}"),
        ("Optical PSF FWHM", f"{2*np.sqrt(2*np.log(2))*sigma:.3f} mm"),
        ("Nyquist frequency", f"{m['nyquist']:.3f} cycles/mm"),
        ("Folded / apparent frequency", f"{m['alias']:.3f} cycles/mm"),
        ("Presampling MTF at target f", f"{m['pre_mtf']:.4f}"),
        ("Predicted continuous contrast after aperture", f"{INPUT_CONTRAST*m['pre_mtf']:.4f}"),
        ("Sampled sinusoid fit contrast", fit_text),
        ("Sample range contrast", f"{m['sample_range_contrast']:.4f}"),
        ("Presampling MTF50", f"{m['mtf50']:.3f} cycles/mm"),
        ("Aperture phase reversal", sign_text)]
    table = "".join(f"<tr><td style='padding:3px 18px 3px 0'>{k}</td><td>{v}</td></tr>" for k,v in rows)
    return f"<div style='border-left:4px solid #197c81;padding:10px 16px'><b>{status}</b><table>{table}</table><small>All contrasts are unitless. A folded component with near-zero contrast may be invisible. MTF50 can exceed Nyquist; it does not imply recoverable sampled detail.</small></div>"

# Close a previous explorer when this cell is rerun.
if "explorer" in globals():
    explorer.close()
style = {"description_width": "170px"}
layout = widgets.Layout(width="min(100%, 650px)")
controls = dict(
    f=widgets.FloatSlider(value=1, min=0.25, max=8, step=0.25,
        description="Frequency (cycles/mm)", readout_format=".2f", continuous_update=False, style=style, layout=layout),
    sigma=widgets.FloatSlider(value=0.04, min=0, max=0.5, step=0.01,
        description="PSF σ (mm)", readout_format=".2f", continuous_update=False, style=style, layout=layout),
    pitch=widgets.SelectionSlider(options=[(f"{p:.2f} mm", p) for p in PITCHES_MM], value=0.10,
        description="Pixel pitch p (mm)", continuous_update=False, style=style, layout=layout),
    phase_deg=widgets.FloatSlider(value=0, min=0, max=360, step=5,
        description="Target phase (degrees)", readout_format=".0f", continuous_update=False, style=style, layout=layout))
metrics_box = widgets.HTML()
plot_image = widgets.Image(format="png", layout=widgets.Layout(width="100%", max_width="1400px"))
updating_preset = False

def refresh(change=None):
    if updating_preset:
        return
    values = {key: control.value for key, control in controls.items()}
    metrics_box.value = metric_html(**values)
    import io
    fig = make_dashboard(**values)
    try:
        buffer = io.BytesIO()
        fig.savefig(buffer, format="png", dpi=110)
        plot_image.value = buffer.getvalue()
    finally:
        plt.close(fig)

PRESETS = {
    "Fine sampling": dict(f=1, sigma=0.04, pitch=0.10, phase_deg=0),
    "Blur limited": dict(f=1, sigma=0.25, pitch=0.10, phase_deg=0),
    "Aliasing": dict(f=3, sigma=0.02, pitch=0.25, phase_deg=0),
    "Nyquist boundary": dict(f=2, sigma=0.02, pitch=0.25, phase_deg=0)}

def set_preset(name):
    global updating_preset
    updating_preset = True
    try:
        for key, value in PRESETS[name].items():
            controls[key].value = value
    finally:
        updating_preset = False
    refresh()

buttons = []
for name in PRESETS:
    button = widgets.Button(description=name, layout=widgets.Layout(width="150px"))
    button.on_click(lambda _, selected=name: set_preset(selected))
    buttons.append(button)
for control in controls.values():
    control.observe(refresh, names="value")
explorer = widgets.VBox([
    widgets.HTML("<b>Choose a starting point, then change one variable at a time.</b>"),
    widgets.Box(buttons, layout=widgets.Layout(display="flex", flex_flow="row wrap")),
    *controls.values(), metrics_box, plot_image])
display(explorer)
refresh()


## 3 · Investigate and explain
Use the metric table as well as the images. Keep contrast fixed and change **one variable at a time**. Record frequency, pitch, blur and phase so someone else can reproduce your observation.

| Trial | $f$ (cycles/mm) | $\sigma$ (mm) | $p$ (mm) | Phase (°) | $f_N$ (cycles/mm) | Folded $f$ (cycles/mm) | MTF at target | What changed? |
|---|---|---|---|---|---|---|---|---|
| Fine sampling | | | | | | | | |
| Blur limited | | | | | | | | |
| Aliasing | | | | | | | | |
| Nyquist: phase 0° | | | | | | | | |
| Nyquist: phase 90° | | | | | | | | |

**1 · Blur and bandwidth.** Compare **Fine sampling** with **Blur limited**. Calculate PSF FWHM for each. Record the MTF at 1 cycle/mm and the MTF50. Explain why optical blur can remove stripe contrast even when many pixels cover each period.

**2 · Pixel area and pitch.** Set $f=1$ cycle/mm, $\sigma=0$, phase $=0°$. Compare $p=0.10$ and $p=0.40$ mm. Calculate $f_N$ and $\mathrm{sinc}(pf)$ for each. Both cases are below Nyquist: why does the larger pixel still reduce contrast? Distinguish pixel-area averaging from sampling.

**3 · A false pattern.** Select **Aliasing**. Use $f_s=1/p$ to calculate the apparent frequency of the 3 cycles/mm target. Verify the result with the profile. Change only pitch to 0.10 mm. Explain why a high fitted contrast in the original coarse detector did not prove that the object was resolved faithfully.

**4 · The Nyquist boundary.** Select **Nyquist boundary** and compare phase 0° with 90°. Explain the change using $x_n=(n+\tfrac12)p$ and the cosine in $d_n$. Why is the fitted amplitude flagged as indeterminate? Can one sample per half-period recover arbitrary sinusoidal phase and amplitude?

**5 · The engineering tradeoff.** Select **Aliasing**, then increase $\sigma$ from 0.02 to 0.15 mm at fixed frequency and pitch. Record the predicted contrast after the aperture at both settings. Explain how blur suppresses the false pattern and what useful detail it also sacrifices. Would making the displayed pixels look smoother restore lost information? Explain using the distinction between interpolation, sampling and optical bandwidth.

Write answers in a saved notebook copy (double-click this text cell) or in your course submission document. Include the completed table or screenshots with visible settings. No code is required.

## Model limits and further reading
This lab isolates spatial resolution: it omits photon noise, finite bit depth, nonuniform sensitivity, real lens aberrations, magnification, and image processing. It uses a single frequency aligned with one detector axis; arbitrary images contain many frequencies and orientations. The intensity model assumes incoherent imaging. The fixed 8 mm window is a crop of an infinite target, not a physical edge in the object.

- [Robertson (2017), *Detector sampling of optical/IR spectra: how many pixels per FWHM?*](https://arxiv.org/abs/1707.06455) discusses finite pixels and sampling phase. Counting pixels across a PSF FWHM is not itself the Nyquist theorem.
- [Jupyter Widgets: Using Interact](https://ipywidgets.readthedocs.io/en/stable/examples/Using%20Interact.html) documents controls and slider updates on release.

**If something stops responding:** reconnect the Colab runtime and choose **Runtime → Run all**. A GitHub preview is static; interaction requires a running Colab or widget-enabled Jupyter session.
